# Module 34 — Exercise 1: Consistent Hashing Ring with Virtual Nodes

Consistent hashing minimizes key redistribution when cache or database servers scale up or down.

In this exercise, you will implement a Consistent Hash Ring in Python using cryptographic MD5 hashes, bisect for binary search on the ring, and virtual nodes for uniform key distribution.

| Detail | Value |
|---|---|
| **Time** | 40 minutes |
| **Prerequisites** | Module 34 README, Module D03 |



# Your turn


### Task 1: Implement `ConsistentHashRing`

Implement `ConsistentHashRing`:
- `add_node(node)`: Add `replicas` virtual node tokens onto the ring.
- `remove_node(node)`: Remove all virtual node tokens for this node.
- `get_node(key)`: Hash `key`, locate the successor token on the ring (wrapping around to index 0 if beyond the end), and return the node name.


In [ ]:
# ANSWER 1
import bisect
import hashlib

class ConsistentHashRing:
    def __init__(self, nodes: list[str] | None = None, replicas: int = 100):
        self.replicas = replicas
        self.ring: list[int] = []
        self.token_to_node: dict[int, str] = {}
        if nodes:
            for n in nodes:
                self.add_node(n)

    def _hash(self, key: str) -> int:
        return int(hashlib.md5(key.encode("utf-8")).hexdigest(), 16)

    def add_node(self, node: str) -> None:
        for i in range(self.replicas):
            token = self._hash(f"{node}#vnode{i}")
            bisect.insort(self.ring, token)
            self.token_to_node[token] = node

    def remove_node(self, node: str) -> None:
        to_remove = [tok for tok, n in self.token_to_node.items() if n == node]
        for tok in to_remove:
            self.ring.remove(tok)
            del self.token_to_node[tok]

    def get_node(self, key: str) -> str | None:
        if not self.ring:
            return None
        key_hash = self._hash(key)
        idx = bisect.bisect_right(self.ring, key_hash)
        if idx == len(self.ring):
            idx = 0
        return self.token_to_node[self.ring[idx]]



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

ring = ConsistentHashRing(["node-a", "node-b", "node-c"], replicas=50)

# Hash 1000 keys and observe distribution
keys = [f"user_session_{i}" for i in range(1000)]
assignments = [ring.get_node(k) for k in keys]
counts = {n: assignments.count(n) for n in ["node-a", "node-b", "node-c"]}
print("Key distribution across 3 nodes:", counts)

# Remove node-b
ring.remove_node("node-b")
assignments_after = [ring.get_node(k) for k in keys]
reassigned_from_b = [a for k, a in zip(keys, assignments_after) if ring.get_node(k) != "node-b"]

results = [
    check(all(250 < count < 450 for count in counts.values()), "Task 1: Virtual nodes evenly distribute 1000 keys across 3 nodes"),
    check("node-b" not in assignments_after, "Task 1: Node removal completely redistributes traffic without crash"),
]
print(f"Summary: {sum(results)}/{len(results)} checks passed.")

